# 02 — Dicionário de Dados & Identificação de Colunas

**Objetivo:** mapear cada coluna do CAGEDMOV ao seu significado e produzir o **dicionário de dados do projeto**.

> ⚠️ Os CSVs em `../dictionaries/` são **sementes** baseadas no layout do Novo CAGED. **Reconcilie-os com o layout oficial** (planilha do MTE) antes de considerar a decodificação final.

In [ ]:
import sys, pathlib
# garante que o pacote caged_etl (em ../) seja importável a partir do notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from caged_etl import setup_logging, get_settings
setup_logging()
settings = get_settings()
print('Municípios:', settings.municipios, '| Anos:', settings.ano_inicio, '-', settings.ano_fim)

## 1. Colunas presentes na camada raw

In [ ]:
import polars as pl
um = sorted(settings.raw_dir.glob('caged_mov_*.parquet'))[0]
df = pl.read_parquet(um)
print('nº de colunas:', len(df.columns))
df.columns

## 2. Domínio de valores das colunas categóricas
Valores distintos observados (ajuda a validar os dicionários).

In [ ]:
from caged_etl.dictionary import DIMENSIONS
for dim in DIMENSIONS:
    if dim.fato_column in df.columns:
        vals = sorted(df[dim.fato_column].unique().to_list(), key=lambda x: (x is None, x))
        print(f'{dim.fato_column:20s} -> {vals[:15]}')

## 3. Dicionários carregados (código → descrição)

In [ ]:
from caged_etl.dictionary import all_dictionaries
dics = all_dictionaries()
for nome, tab in dics.items():
    print(f'### {nome} ({tab.height} códigos)')
    print(tab)


## 4. Dicionário de dados do projeto (coluna → tipo → domínio)

In [ ]:
from caged_etl.config import CURATED_COLUMNS
linhas = []
for col in CURATED_COLUMNS:
    if col in df.columns:
        n_distintos = df[col].n_unique()
        linhas.append({'coluna': col, 'tipo_origem': str(df[col].dtype),
                       'valores_distintos': n_distintos})
dicionario_projeto = pl.DataFrame(linhas)
dicionario_projeto